# qcd-einsum 入门（`pyqcd.einsum`）

`pyqcd.einsum`（下称 **qcd-einsum**）是本仓库自带的收缩引擎：

* **接口与 `opt_einsum` 兼容** —— `contract` / `contract_path` / `contract_expression` /
  `shared_intermediates` / `paths` / `backends` / `PathInfo`，可以直接替换使用；
* **纯 Python 规划 + 可选原生执行** —— 规划层（`_plan_py.py`）永远可用，
  原生层分 **KUPL（CPU）** 与 **CUDA（GPU）** 两套，缺少 `.so` 时自动回退，结果不变；
* **方程形态特化** —— 对格点 QCD 常见收缩形态自动匹配专用执行路径，命中时打印提示，
  并可用参数关闭以便做 A/B 对照。

本 notebook 按"能用 → 快不快 → 稳不稳"的顺序讲解，全部单元都可直接运行。

## 1. 环境与构建标识

In [1]:
import os
import sys
import time
import statistics

REPO = os.path.abspath(os.path.join(os.getcwd()))
while REPO != os.path.dirname(REPO) and not os.path.isdir(os.path.join(REPO, 'pyqcd')):
    REPO = os.path.dirname(REPO)
sys.path.insert(0, REPO)

import numpy as np
import pyqcd.einsum as pe

print('repo           :', REPO)
print('numpy          :', np.__version__)
print('pyqcd.einsum   :', pe.__version__ if hasattr(pe, '__version__') else '0.1.0')

info = pe.build_info()
for key, value in info.items():
    print(f'{key:15s}:', value)

repo           : /home/zhangxin/PyQCD
numpy          : 2.5.4
pyqcd.einsum   : 0.1.0
backend        : cuda
mpi            : False
kupl_mpi_build : False
mpi_label      : 非 MPI 版本（CUDA 后端未实现 MPI 分片）
native_library : /home/zhangxin/PyQCD/pyqcd/einsum/cuda/libpyqcdeinsum_cuda.so


`build_info()` 是判断"当前到底装了什么"的权威入口：

| 字段 | 含义 |
|---|---|
| `backend` | 当前可用的原生后端：`cuda` / `kupl` / `python`（后者表示纯 Python 回退） |
| `mpi` / `kupl_mpi_build` | 当前 KUPL 二进制是否编入真实 MPI 分片 |
| `mpi_label` | 人类可读标识：`MPI 版本` / `非 MPI 版本` / `非 MPI 版本（CUDA 后端未实现 MPI 分片）` |
| `native_library` | 实际加载的原生库路径 |

## 2. 最基本的用法：`contract`

In [2]:
rng = np.random.default_rng(20261108)

a = (rng.standard_normal((3, 4)) + 1j * rng.standard_normal((3, 4))).astype(np.complex128)
b = (rng.standard_normal((4, 5)) + 1j * rng.standard_normal((4, 5))).astype(np.complex128)

qcd = pe.contract('ij,jk->ik', a, b)
ref = a @ b
print('shape      :', qcd.shape, qcd.dtype)
print('max rel err:', float(np.max(np.abs(qcd - ref)) / np.max(np.abs(ref))))

shape      : (3, 5) complex128
max rel err: 0.0


In [3]:
# 隐式输出、迹、Hadamard、ellipsis 都支持
print(pe.contract('ij,jk', a, b).shape)          # 隐式输出 -> ik
print(pe.contract('ii->', np.eye(4, dtype=np.complex128)))   # 迹
print(pe.contract('ij,ij->ij', a, a).shape)      # Hadamard
print(pe.contract('...ij,...jk->...ik',
                  np.ones((2, 3, 4), np.complex128),
                  np.ones((2, 4, 5), np.complex128)).shape)

(3, 5)
(4+0j)
(3, 4)
(2, 3, 5)


/home/zhangxin/PyQCD/pyqcd/einsum/_api.py:1129: RuntimeWarning: pyqcd.einsum native plan unavailable; using Python execution: native execution does not support repeated axes in 'ii'
  handled, native_result = _try_native_contract(


## 3. 规划：`contract_path` 与 `PathInfo`

In [4]:
path, info = pe.contract_path('ij,jk,kl->il', a, b, b.T.copy())
print('path        :', path)
print('opt_cost    :', int(info.opt_cost))
print('naive_cost  :', int(info.naive_cost))
print('speedup     :', float(info.speedup))
print('largest_int :', int(info.largest_intermediate))
print('字段齐全    :', all(hasattr(info, f) for f in (
    'contraction_list', 'input_subscripts', 'output_subscript', 'path', 'indices',
    'scale_list', 'naive_cost', 'opt_cost', 'speedup', 'size_list', 'size_dict',
    'shapes', 'eq', 'largest_intermediate')))

path        : [(0, 1), (0, 1)]
opt_cost    : 240
naive_cost  : 720
speedup     : 3.0
largest_int : 15
字段齐全    : True


In [5]:
# 只给形状也能规划（不做任何计算）
_, shape_info = pe.contract_path('ij,jk->ik', (1000, 1000), (1000, 1000),
                                 shapes=True, optimize='dp')
print('shapes=True opt_cost =', int(shape_info.opt_cost))

shapes=True opt_cost = 2000000000


## 4. 可复用表达式：`contract_expression`

In [6]:
expr = pe.contract_expression('ij,jk->ik', (3, 4), (4, 5), optimize='dp')

for _ in range(3):
    a2 = (rng.standard_normal((3, 4)) + 1j * rng.standard_normal((3, 4))).astype(np.complex128)
    b2 = (rng.standard_normal((4, 5)) + 1j * rng.standard_normal((4, 5))).astype(np.complex128)
    got = expr(a2, b2)
    assert np.allclose(got, a2 @ b2), 'expression 结果不一致'
print('contract_expression 复用 3 次，结果全部正确')

contract_expression 复用 3 次，结果全部正确


## 5. 共享中间结果：`shared_intermediates`

In [7]:
with pe.shared_intermediates() as cache:
    pe.contract('ab,bc->ac', a, b, backend='numpy')
    pe.contract('ab,bc->ac', a, b, backend='numpy')      # 命中缓存
    print('cache after repeat:', dict(pe.count_cached_ops(cache)))

cache after repeat: {'tensor': 3, 'tensordot': 1, 'transpose': 1}


共享缓存按**对象身份**做键：重复的子表达式只算一次。
注意它只在上下文内生效，且缓存的是张量本身，不会缓存"输入内容变更"后的结果。

## 6. 方程形态特化

In [8]:
for name, kind, equation in pe.specializations():
    print(f'{name:24s} {kind:28s} {equation[:60]}')

nucleon-2pt(E1)          fused(needs-optimization)    gahb,mcnd,oepf,ac,og,Mbdf,Mnph,em->M
nucleon-3pt(E2)          plan-cache                   Geifj,ckdl,agbh,ce,gi,Mbdf,Mhjl,Gka->GM
gauge-propagator(E4)     fused                        zyxcTabn,zyxgTdef,Gae,Tzyxgc->TGdbfn
gamma-sandwich(E5)       fused                        ab,...bce,cd->...dae
sush-2pt(S1)             plan-cache                   eofp,ambn,cgdh,ce,mo,Nbdf,Mnph,ga->MN
sush-3pt-munu(S2)        plan-cache                   gahb,menf,codp,bd,Mfn,Mph,ac,MGem,MGog->MG
sush-2pt-mu(S3)          plan-cache                   manb,cedf,bd,Mfn,ac,MFem->MF
sush-3pt-main(P1)        fused(needs-optimization)    agbh,eqfr,ospt,cmdn,ikjl,ce,Gmo,gi,kq,Nbdf,NLnp,Mhj,Mlrt,ax,
sush-3pt-main(P2)        fused(needs-optimization)    agbh,eqfr,ospt,ckdl,imjn,ce,Gmo,gi,kq,Nbdf,NLnp,Mhj,Mlrt,ax,


`kind` 的取值含义：

| kind | 含义 |
|---|---|
| `fused` | 有真正融合的专用 kernel（例如 γ 夹心、规范场收缩） |
| `fused(needs-optimization)` | 专用 kernel 已接线，但相对通用路径的收益仍不稳定，仍在优化 |
| `plan-cache` | 只复用计划缓存，**没有**专用 kernel |
| `multi-step` | 只是形态标记（多步收缩） |

In [9]:
# 命中特化时，库会打印一行提示（每个形态每进程一次）
eq = 'gahb,mcnd,oepf,ac,og,Mbdf,Mnph,em->M'     # 核子 2pt（E1）

nvec, mom = 8, 1
peram = (rng.standard_normal((4, 4, nvec, nvec))
         + 1j * rng.standard_normal((4, 4, nvec, nvec))).astype(np.complex128)
gamma = (rng.standard_normal((4, 4)) + 1j * rng.standard_normal((4, 4))).astype(np.complex128)
vvv = (rng.standard_normal((mom, nvec, nvec, nvec))
       + 1j * rng.standard_normal((mom, nvec, nvec, nvec))).astype(np.complex128)
projector = np.eye(4, dtype=np.complex128)

operands = [peram, peram, peram, gamma, gamma, vvv, vvv, projector]

on = pe.contract(eq, *operands, optimize='dp')                    # 允许特化
off = pe.contract(eq, *operands, optimize='dp', specialize=False)  # 强制通用路径
print('specialize 开/关结果一致:', bool(np.allclose(on, off)))

[pyqcd.einsum] 专用路径: nucleon-2pt(E1) [fused(needs-optimization)] 8 算子核子 2pt；KUPL/CUDA 专用 kernel 均已接线（KUPL 1.1–1.4x；CUDA 0.86–1.17x 待优化）


specialize 开/关结果一致: True


In [10]:
# 小规模 A/B 计时（严格对比请看 bench_native，本处只做量级演示）
def timed(fn, n=10, warmup=2):
    for _ in range(warmup):
        fn()
    samples = []
    for _ in range(n):
        t0 = time.perf_counter()
        fn()
        samples.append(time.perf_counter() - t0)
    return statistics.median(samples) * 1e3

print(f"specialize=True : {timed(lambda: pe.contract(eq, *operands, optimize='dp')):8.3f} ms")
print(f"specialize=False: {timed(lambda: pe.contract(eq, *operands, optimize='dp', specialize=False)):8.3f} ms")

specialize=True :    0.281 ms


specialize=False:    3.983 ms


运行时开关（进程级）：

| 变量 | 作用 |
|---|---|
| `PYQCD_EINSUM_SPECIALIZE=0` | 关闭全部方程特化 |
| `PYQCD_EINSUM_DISABLE_FUSED=1` | 关闭原生融合内核（强制通用多步） |
| `PYQCD_EINSUM_KERNEL_KUPL=0` / `PYQCD_EINSUM_KERNEL_CUDA=0` | 只关闭指定后端的专用 kernel |

## 7. 原生后端与回退不变量

In [11]:
from pyqcd.einsum import _native

print('原生后端选择 :', _native.backend_name())
print('原生库路径   :', _native.library_path())
print('MPI 标识     :', pe.build_info()['mpi_label'])

原生后端选择 : cuda
原生库路径   : /home/zhangxin/PyQCD/pyqcd/einsum/cuda/libpyqcdeinsum_cuda.so
MPI 标识     : 非 MPI 版本（CUDA 后端未实现 MPI 分片）


三条不变量（改动代码时必须遵守）：

1. 缺 `.so` → 自动使用纯 Python 规划（`_plan_py.py`），**结果不变、仅变慢**；
2. `_native.py` 只做 `ctypes`，import 时不加载库、不编译、不报错；
3. 任何原生路径失败 → 只 warning 一次并回退，**不把异常抛给调用方**。

想确认原生路径是否真的被用上，用仓库的两条命令：

```bash
python -m pyqcd.testing.einsum.verify_native --backend kupl
python -m pyqcd.testing.einsum.verify_native --backend cuda
```

## 8. 正确性：与 `opt_einsum` 和 `numpy` 对照

In [12]:
import opt_einsum as oe

def random_case(rng, nops_max=4):
    symbols = 'abcdefgh'
    pool = list(symbols[:int(rng.integers(2, 6))])   # numpy 2.x 的 choice 不接受 str
    size = {label: int(rng.choice([1, 2, 3])) for label in pool}
    terms = []
    for _ in range(int(rng.integers(2, nops_max + 1))):
        width = int(rng.integers(1, min(4, len(pool)) + 1))
        chosen = list(rng.choice(pool, size=width, replace=False))
        if rng.random() < 0.2:
            chosen.append(chosen[0])
        terms.append(''.join(chosen))
    used = sorted(set(''.join(terms)))
    out = ''.join(label for label in used if rng.random() < 0.6)
    equation = ','.join(terms) + '->' + out
    arrays = [(rng.standard_normal(tuple(size[c] for c in term))
               + 1j * rng.standard_normal(tuple(size[c] for c in term))
               ).astype(np.complex128) for term in terms]
    return equation, arrays

bad = 0
for index in range(20):
    case_rng = np.random.default_rng(1000 + index)
    equation, arrays = random_case(case_rng)
    reference = np.einsum(equation, *arrays, optimize=False)
    got_qcd = pe.contract(equation, *arrays, optimize='greedy')
    got_opt = oe.contract(equation, *arrays, optimize='greedy')
    for label, value in (('qcd', got_qcd), ('opt', got_opt)):
        denom = max(float(np.max(np.abs(reference))), 1e-300)
        rel = float(np.max(np.abs(value - reference)) / denom)
        if rel > 1e-12:
            bad += 1
            print('MISMATCH', label, equation, rel)
print(f'20 个随机方程 × 2 个实现，超出容差 {bad} 项')

20 个随机方程 × 2 个实现，超出容差 0 项


## 9. 性能：怎么测才算数

严格基准必须走 `bench_native`（它在 **导入 numpy 之前**钉住线程数，并做进程隔离）：

```bash
PYQCD_EINSUM_ORACLE=installed PYQCD_EINSUM_NATIVE=kupl PYQCD_BENCH_THREADS=4 \
  python -m pyqcd.testing.einsum.bench_native \
  --data synthetic --profiles T1 --backends cpu --iters 10 --warmup 4 --planner production
```

读数字时的四条硬规则（都是踩过坑之后总结的）：

1. **预热要够**：CUDA 前 3–4 次调用会因显存池增长慢 3–4 倍，默认预热已经改为 4 轮；
2. **对照要同口径**：比较"专用 vs 通用"时，通用一侧必须显式 `specialize=False`，
   否则接线的 kernel 会被当成通用路径；
3. **别在并发环境测**：同配置在后台有任务时能差 3.7 倍；正式数字要独占机器；
4. **短任务不算数**：双方都低于 100 ms 的单元格记为 `too_short`，不参与倍率判定。

In [13]:
# 这里只做一次"量级"演示：真实结论请用上面的 bench_native
shapes = [(64, 64), (64, 64)]
x = (rng.standard_normal(shapes[0]) + 1j * rng.standard_normal(shapes[0])).astype(np.complex128)
y = (rng.standard_normal(shapes[1]) + 1j * rng.standard_normal(shapes[1])).astype(np.complex128)

print(f"pyqcd : {timed(lambda: pe.contract('ij,jk->ik', x, y, optimize='dp')):8.3f} ms")
print(f"opt   : {timed(lambda: oe.contract('ij,jk->ik', x, y, optimize='dp')):8.3f} ms")
print(f"numpy : {timed(lambda: np.einsum('ij,jk->ik', x, y)):8.3f} ms")

pyqcd :    0.115 ms
opt   :    0.068 ms


numpy :    0.745 ms


## 10. 测试入口

In [14]:
import os

base = os.path.join(REPO, 'pyqcd', 'testing', 'einsum')
for name in sorted(os.listdir(base)):
    if name.endswith('.py'):
        print(name)

__init__.py
_oracle.py
_target.py
bench_cuda_mpi.py
bench_distillation.py
bench_native.py
verify_compat.py
verify_cuda.py
verify_cuda_mpi.py
verify_mpi.py
verify_multiprocess.py
verify_native.py


| 入口 | 用途 |
|---|---|
| `python -m pyqcd.testing.einsum.verify_native --backend kupl\\|cuda` | 原生内核正确性（四 dtype、chain、batch、unary、dot、sandwich、compiled context） |
| `python -m pyqcd.testing.einsum.verify_compat` | 与 `opt_einsum` 的 980 项差分（需要 pytest；子模块不可用时设 `PYQCD_EINSUM_ORACLE=installed`） |
| `python -m pyqcd.testing.einsum.bench_native ...` | 严格性能基准 |
| `bash pyqcd/testing/einsum/simple/run-all.sh` | 子系统一键套件（compat/memory/equations/native/kernels） |
| `bash pyqcd/testing/simple/run-all.sh` | 逐函数 API 对照套件（参考实现为 `torch.einsum`） |
| `python -m pyqcd.testing.einsum.simple.workflow` | 生产工作流时间占比模型 |
| `python -m pyqcd.testing.einsum.simple.kernel_acceptance` | 专用 kernel 验收（慢于 opt_einsum 即判 BUG） |

## 11. 已知限制（不要当成 bug 提）

| 项 | 现状 |
|---|---|
| `contract_expression` | 结果正确，但**不走专用 kernel**（只有 `contract` 会做形态分派） |
| MPI | 只有 KUPL 有真实实现（单输出轴分片 + `Allgatherv`，无跨 rank 归约）；CUDA 侧是 fail-closed stub |
| 小规模 | 固定开销占主导，倍率没有参考意义 |
| `needs-optimization` 标记 | 相关 kernel 已接线，但收益仍随通用路径波动，仍在优化 |

## 小结

* 把它当作 `opt_einsum` 的替代品即可：同签名、同语义、同样的 `PathInfo`；
* 原生后端是可选加速层，缺失或失败都会**静默回退**，不会改变数值结果；
* 方程特化是"锦上添花"：可用 `specialize=False` 或环境变量随时关掉做对照；
* 任何性能结论都必须来自 `bench_native`，且满足预热、同口径、独占、非短任务四条规则。